# Artifact publication repository
Immutable staged generations, explicit source identity, pinned reads and safe maintenance.

In [ ]:
#| default_exp core.artifacts


In [ ]:
#| export
from __future__ import annotations

from contextlib import contextmanager
from dataclasses import asdict, dataclass, field
import json
import os
from pathlib import Path
import sqlite3
import tempfile
import time
import uuid
from typing import Any

from ribosome.core.document import Document, DocumentGraph, Node, canonical_json, content_hash


@dataclass(frozen=True)
class Publication:
    generation: str
    document_id: str
    revision_id: str
    graph_hash: str
    state: str
    previous_generation: str | None
    created_at: float
    settings: dict[str, Any] = field(default_factory=dict)
    completeness: dict[str, Any] = field(default_factory=dict)


@dataclass(frozen=True)
class CorpusManifest:
    corpus_id: str
    document_ids: tuple[str, ...]
    tombstones: tuple[str, ...] = ()
    authoritative: bool = False
    complete: bool = False


class PublicationConflict(RuntimeError):
    pass


class ArtifactSnapshot:
    def __init__(self, repository, generations):
        self.repository = repository
        self.generations = dict(generations)

    def read(self, document_id):
        return self.repository._read_generation(self.generations[document_id])

    def nodes(self, *, kinds=(), node_ids=(), limit=100):
        return self.repository._query_nodes(tuple(self.generations.values()), kinds=kinds,
                                            node_ids=node_ids, limit=limit)


class ArtifactRepository:
    """Single-writer staged publication with immutable blobs and reader pins.

    Preparing a generation never switches visibility. A durable SQLite transaction
    is the commit boundary. Recovery checks already committed content, and never
    interprets an incomplete preparation as a request to regenerate summaries.
    """

    def __init__(self, root, *, fault_hook=None):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)
        self.blobs = self.root / 'blobs'
        self.blobs.mkdir(exist_ok=True)
        self.path = self.root / 'catalog.sqlite3'
        self.fault_hook = fault_hook
        with self._connection() as connection:
            connection.executescript('''
                CREATE TABLE IF NOT EXISTS documents (
                    id TEXT PRIMARY KEY, corpus_id TEXT NOT NULL, metadata TEXT NOT NULL,
                    tombstoned INTEGER NOT NULL DEFAULT 0);
                CREATE TABLE IF NOT EXISTS aliases (
                    alias TEXT PRIMARY KEY, document_id TEXT NOT NULL REFERENCES documents(id));
                CREATE TABLE IF NOT EXISTS publications (
                    generation TEXT PRIMARY KEY, document_id TEXT NOT NULL,
                    revision_id TEXT NOT NULL, graph_hash TEXT NOT NULL,
                    state TEXT NOT NULL, previous_generation TEXT, created_at REAL NOT NULL,
                    committed_at REAL, settings TEXT NOT NULL, completeness TEXT NOT NULL);
                CREATE INDEX IF NOT EXISTS publication_document ON publications(document_id, state, created_at);
                CREATE TABLE IF NOT EXISTS active_publications (
                    document_id TEXT PRIMARY KEY, generation TEXT NOT NULL UNIQUE REFERENCES publications(generation));
                CREATE TABLE IF NOT EXISTS nodes (
                    generation TEXT NOT NULL REFERENCES publications(generation), id TEXT NOT NULL,
                    kind TEXT NOT NULL, parent_id TEXT, position INTEGER NOT NULL,
                    payload_hash TEXT NOT NULL, PRIMARY KEY(generation,id));
                CREATE INDEX IF NOT EXISTS node_kind ON nodes(generation,kind,position);
                CREATE TABLE IF NOT EXISTS reader_pins (
                    reader TEXT NOT NULL, generation TEXT NOT NULL, created_at REAL NOT NULL,
                    PRIMARY KEY(reader,generation));
                CREATE TABLE IF NOT EXISTS lifecycle_events (
                    created_at REAL NOT NULL, action TEXT NOT NULL, payload TEXT NOT NULL);
            ''')

    @contextmanager
    def _connection(self):
        connection = sqlite3.connect(self.path, timeout=30)
        connection.row_factory = sqlite3.Row
        connection.execute('PRAGMA foreign_keys=ON')
        connection.execute('PRAGMA journal_mode=WAL')
        connection.execute('PRAGMA synchronous=FULL')
        try:
            yield connection
            connection.commit()
        except BaseException:
            connection.rollback()
            raise
        finally:
            connection.close()

    def _fault(self, boundary):
        if self.fault_hook:
            self.fault_hook(boundary)

    def _write_blob(self, value):
        data = canonical_json(value).encode('utf-8')
        digest = content_hash(data)
        path = self.blobs / (digest + '.json')
        if path.exists():
            if content_hash(path.read_bytes()) != digest:
                raise ValueError(f'Corrupt existing blob {digest}')
            return digest
        temporary = None
        try:
            with tempfile.NamedTemporaryFile(dir=self.blobs, delete=False) as stream:
                temporary = Path(stream.name)
                stream.write(data)
                stream.flush()
                os.fsync(stream.fileno())
            os.replace(temporary, path)
            if hasattr(os, 'O_DIRECTORY'):
                descriptor = os.open(self.blobs, os.O_RDONLY | os.O_DIRECTORY)
                try:
                    os.fsync(descriptor)
                finally:
                    os.close(descriptor)
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)
        return digest

    def _read_blob(self, digest):
        if len(digest) != 64 or any(c not in '0123456789abcdef' for c in digest):
            raise ValueError('Invalid content address')
        data = (self.blobs / (digest + '.json')).read_bytes()
        if content_hash(data) != digest:
            raise ValueError(f'Blob integrity failure: {digest}')
        return json.loads(data)

    def adopt_document(self, source_uri, *, document_id=None, title='', corpus_id='default', manual_id=''):
        """Assign once, or explicitly add a new alias to an existing identity."""
        source_uri = str(source_uri)
        with self._connection() as connection:
            connection.execute('BEGIN IMMEDIATE')
            row = connection.execute('SELECT d.metadata FROM aliases a JOIN documents d ON d.id=a.document_id WHERE a.alias=?', (source_uri,)).fetchone()
            if row:
                document = Document(**json.loads(row['metadata']))
                if document_id and document.id != document_id:
                    raise ValueError('Source alias already belongs to another document')
                return document
            identity = document_id or str(uuid.uuid4())
            row = connection.execute('SELECT metadata FROM documents WHERE id=?', (identity,)).fetchone()
            if row:
                old = Document(**json.loads(row['metadata']))
                if old.corpus_id != corpus_id:
                    raise ValueError('Document corpus mismatch')
                document = Document(old.id, old.title, old.corpus_id, old.manual_id,
                                    tuple(dict.fromkeys([*old.aliases, source_uri])))
            else:
                document = Document(identity, title, corpus_id, manual_id, (source_uri,))
            connection.execute('INSERT INTO documents(id,corpus_id,metadata) VALUES(?,?,?) ON CONFLICT(id) DO UPDATE SET metadata=excluded.metadata',
                               (identity, corpus_id, canonical_json(asdict(document))))
            connection.execute('INSERT INTO aliases VALUES(?,?)', (source_uri, identity))
            return document

    def relocate_document(self, document_id, new_uri):
        with self._connection() as connection:
            row = connection.execute('SELECT metadata FROM documents WHERE id=?', (document_id,)).fetchone()
            if not row:
                raise KeyError(document_id)
            document = Document(**json.loads(row['metadata']))
        return self.adopt_document(new_uri, document_id=document_id, corpus_id=document.corpus_id)

    @staticmethod
    def _publication(row):
        return Publication(row['generation'], row['document_id'], row['revision_id'], row['graph_hash'],
                           row['state'], row['previous_generation'], row['created_at'],
                           json.loads(row['settings']), json.loads(row['completeness']))

    def prepare(self, graph: DocumentGraph, *, settings=None, completeness=None) -> Publication:
        graph.validate()
        generation = uuid.uuid4().hex
        created = time.time()
        with self._connection() as connection:
            connection.execute('BEGIN IMMEDIATE')
            known = connection.execute('SELECT corpus_id FROM documents WHERE id=?', (graph.document.id,)).fetchone()
            if known and known['corpus_id'] != graph.document.corpus_id:
                raise ValueError('Logical document corpus changed')
            connection.execute('INSERT OR IGNORE INTO documents(id,corpus_id,metadata) VALUES(?,?,?)',
                               (graph.document.id, graph.document.corpus_id, canonical_json(asdict(graph.document))))
            for alias in graph.document.aliases:
                owner = connection.execute('SELECT document_id FROM aliases WHERE alias=?', (alias,)).fetchone()
                if owner and owner['document_id'] != graph.document.id:
                    raise ValueError('Alias belongs to another document')
                connection.execute('INSERT OR IGNORE INTO aliases VALUES(?,?)', (alias, graph.document.id))
            previous = connection.execute('SELECT generation FROM active_publications WHERE document_id=?', (graph.document.id,)).fetchone()
            previous = previous['generation'] if previous else None
            digest = self._write_blob(graph.to_dict())
            node_blobs = [(node, self._write_blob(asdict(node))) for node in graph.nodes]
            self._fault('blobs_written')
            connection.execute('INSERT INTO publications VALUES(?,?,?,?,?,?,?,?,?,?)',
                               (generation, graph.document.id, graph.revision.id, digest, 'prepared', previous,
                                created, None, canonical_json(settings or {}), canonical_json(completeness or {})))
            connection.executemany('INSERT INTO nodes VALUES(?,?,?,?,?,?)',
                                   [(generation, node.id, node.kind, node.parent_id, i, blob)
                                    for i, (node, blob) in enumerate(node_blobs)])
            self._fault('preparation_recorded')
        self._fault('prepared')
        return Publication(generation, graph.document.id, graph.revision.id, digest, 'prepared', previous,
                           created, settings or {}, completeness or {})

    def commit(self, generation, *, expected_previous=...) -> Publication:
        with self._connection() as connection:
            connection.execute('BEGIN IMMEDIATE')
            row = connection.execute('SELECT * FROM publications WHERE generation=?', (generation,)).fetchone()
            if not row:
                raise KeyError(generation)
            publication = self._publication(row)
            if publication.state == 'committed':
                return publication
            if publication.state != 'prepared':
                raise ValueError('Only prepared publications can commit')
            graph = DocumentGraph.from_dict(self._read_blob(publication.graph_hash))
            if graph.document.id != publication.document_id or graph.revision.id != publication.revision_id:
                raise ValueError('Publication graph identity mismatch')
            if publication.completeness.get('missing'):
                raise ValueError('Cannot publish incomplete required work')
            current = connection.execute('SELECT generation FROM active_publications WHERE document_id=?', (publication.document_id,)).fetchone()
            current = current['generation'] if current else None
            expected = publication.previous_generation if expected_previous is ... else expected_previous
            if current != expected:
                raise PublicationConflict('Active publication changed during preparation')
            self._fault('before_pointer')
            connection.execute('INSERT INTO active_publications VALUES(?,?) ON CONFLICT(document_id) DO UPDATE SET generation=excluded.generation',
                               (publication.document_id, generation))
            connection.execute("UPDATE publications SET state='committed',committed_at=? WHERE generation=?", (time.time(), generation))
            self._fault('pointer_written')
        # After this boundary the generation is authoritative; failures are recovery work.
        self._fault('committed')
        return Publication(**{**asdict(publication), 'state': 'committed'})

    def _read_generation(self, generation):
        with self._connection() as connection:
            row = connection.execute('SELECT graph_hash,state FROM publications WHERE generation=?', (generation,)).fetchone()
            if not row or row['state'] != 'committed':
                raise KeyError(f'No committed publication {generation}')
        return DocumentGraph.from_dict(self._read_blob(row['graph_hash']))

    @contextmanager
    def snapshot(self, document_ids=None, *, generations=None):
        reader = uuid.uuid4().hex
        with self._connection() as connection:
            connection.execute('BEGIN IMMEDIATE')
            if generations is None:
                sql = 'SELECT a.document_id,a.generation FROM active_publications a JOIN documents d ON d.id=a.document_id WHERE d.tombstoned=0'
                params = []
                if document_ids is not None:
                    ids = tuple(document_ids)
                    sql += ' AND a.document_id IN (' + ','.join('?' for _ in ids) + ')'
                    params.extend(ids)
                selected = dict(connection.execute(sql, params).fetchall())
            else:
                selected = {}
                for generation in generations:
                    row = connection.execute("SELECT document_id FROM publications WHERE generation=? AND state='committed'", (generation,)).fetchone()
                    if not row:
                        raise KeyError(generation)
                    if row['document_id'] in selected:
                        raise ValueError('Snapshot cannot mix document revisions')
                    selected[row['document_id']] = generation
            connection.executemany('INSERT INTO reader_pins VALUES(?,?,?)',
                                   [(reader, generation, time.time()) for generation in selected.values()])
        try:
            yield ArtifactSnapshot(self, selected)
        finally:
            with self._connection() as connection:
                connection.execute('DELETE FROM reader_pins WHERE reader=?', (reader,))

    def read(self, document_id, generation=None):
        with self.snapshot([document_id], generations=[generation] if generation else None) as snapshot:
            return snapshot.read(document_id)

    def _query_nodes(self, generations, *, kinds=(), node_ids=(), limit=100):
        if not isinstance(limit, int) or limit < 1:
            raise ValueError('A positive bounded limit is required')
        if not generations:
            return []
        sql = 'SELECT payload_hash FROM nodes WHERE generation IN (' + ','.join('?' for _ in generations) + ')'
        params = list(generations)
        for name, values in [('kind', kinds), ('id', node_ids)]:
            if values:
                sql += f' AND {name} IN (' + ','.join('?' for _ in values) + ')'
                params.extend(values)
        sql += ' ORDER BY generation,position LIMIT ?'
        params.append(limit)
        with self._connection() as connection:
            hashes = [r['payload_hash'] for r in connection.execute(sql, params)]
        return [Node(**self._read_blob(digest)) for digest in hashes]

    def recover(self):
        """Report recoverable preparations and verify committed publications."""
        report = {'committed': [], 'prepared': [], 'invalid': []}
        with self._connection() as connection:
            rows = connection.execute('SELECT * FROM publications ORDER BY created_at').fetchall()
        for row in rows:
            try:
                graph = DocumentGraph.from_dict(self._read_blob(row['graph_hash']))
                if graph.document.id != row['document_id'] or graph.revision.id != row['revision_id']:
                    raise ValueError('Publication identity mismatch')
                report[row['state']].append(row['generation'])
            except (ValueError, OSError, KeyError) as error:
                report['invalid'].append({'generation': row['generation'], 'error': str(error)})
        return report

    def rollback(self, document_id, generation):
        with self._connection() as connection:
            connection.execute('BEGIN IMMEDIATE')
            row = connection.execute("SELECT graph_hash FROM publications WHERE document_id=? AND generation=? AND state='committed'", (document_id, generation)).fetchone()
            if not row:
                raise ValueError('Rollback requires a committed generation of this document')
            DocumentGraph.from_dict(self._read_blob(row['graph_hash']))
            connection.execute('INSERT INTO active_publications VALUES(?,?) ON CONFLICT(document_id) DO UPDATE SET generation=excluded.generation', (document_id, generation))
            connection.execute('INSERT INTO lifecycle_events VALUES(?,?,?)', (time.time(), 'rollback', canonical_json({'document_id': document_id, 'generation': generation})))

    def synchronize_manifest(self, manifest: CorpusManifest, *, apply=False):
        if not manifest.authoritative or not manifest.complete:
            raise ValueError('Synchronization requires a complete authoritative manifest')
        if set(manifest.document_ids) & set(manifest.tombstones):
            raise ValueError('A document cannot be active and tombstoned')
        with self._connection() as connection:
            connection.execute('BEGIN IMMEDIATE')
            known = {row['id'] for row in connection.execute('SELECT id FROM documents WHERE corpus_id=?', (manifest.corpus_id,))}
            requested = set(manifest.document_ids) | set(manifest.tombstones)
            if not requested <= known:
                raise ValueError('Manifest includes unknown or other-corpus document IDs')
            result = {'tombstones': sorted(set(manifest.tombstones)), 'retained_unlisted': sorted(known - requested), 'applied': apply}
            if apply:
                connection.executemany('UPDATE documents SET tombstoned=1 WHERE id=?', [(i,) for i in manifest.tombstones])
                connection.executemany('UPDATE documents SET tombstoned=0 WHERE id=?', [(i,) for i in manifest.document_ids])
                connection.execute('INSERT INTO lifecycle_events VALUES(?,?,?)', (time.time(), 'synchronize', canonical_json(asdict(manifest))))
            return result

    def cleanup(self, *, dry_run=True, grace_seconds=7 * 86400, keep_previous=1):
        if grace_seconds < 0 or keep_previous < 1:
            raise ValueError('Cleanup requires nonnegative grace and at least one rollback generation')
        with self._connection() as connection:
            connection.execute('BEGIN IMMEDIATE')
            rows = connection.execute('SELECT * FROM publications ORDER BY created_at DESC').fetchall()
            protected = {r[0] for r in connection.execute('SELECT generation FROM active_publications')}
            protected.update(r[0] for r in connection.execute('SELECT generation FROM reader_pins'))
            protected.update(r['generation'] for r in rows if r['state'] != 'committed')
            active = set(protected)
            kept = {}
            for row in rows:
                doc = row['document_id']
                if row['generation'] not in active and row['state'] == 'committed' and kept.get(doc, 0) < keep_previous:
                    protected.add(row['generation'])
                    kept[doc] = kept.get(doc, 0) + 1
            cutoff = time.time() - grace_seconds
            candidates = [r['generation'] for r in rows if r['generation'] not in protected and r['created_at'] < cutoff]
            if not dry_run:
                for generation in candidates:
                    connection.execute('DELETE FROM nodes WHERE generation=?', (generation,))
                    connection.execute('DELETE FROM publications WHERE generation=?', (generation,))
            if not dry_run:
                # Commit catalog removal first: a later failure cannot resurrect references
                # to blobs already removed. Reacquire the writer lock before collecting.
                connection.commit()
                self._fault('cleanup_catalog_committed')
                connection.execute('BEGIN IMMEDIATE')
            # File deletion stays under the writer lock, excluding concurrent preparation.
            referenced = {r[0] for r in connection.execute('SELECT graph_hash FROM publications')}
            referenced.update(r[0] for r in connection.execute('SELECT payload_hash FROM nodes'))
            orphans = [p for p in self.blobs.glob('*.json') if p.stem not in referenced and p.stat().st_mtime < cutoff]
            if not dry_run:
                for path in orphans:
                    path.unlink()
            return {'dry_run': dry_run, 'generations': candidates,
                    'orphan_blobs': [p.name for p in orphans], 'protected_generations': sorted(protected)}

    def stats(self):
        with self._connection() as connection:
            report = {name: connection.execute(f'SELECT COUNT(*) FROM {name}').fetchone()[0]
                      for name in ('documents', 'publications', 'active_publications', 'nodes', 'reader_pins')}
        report['blob_bytes'] = sum(p.stat().st_size for p in self.blobs.glob('*.json'))
        report['catalog_bytes'] = self.path.stat().st_size
        return report
